# LlamaIndex workflows with typed events and state

An original, deterministic LlamaIndex companion lesson.

## Learning objectives

- Represent start, classification, response, retry, and stop as typed events.
- Implement branching and bounded loop termination.
- Record shared state and an exact workflow trace.

## Setup

The offline workflow runner uses `dataclasses` and deterministic steps. It demonstrates event routing without an event loop, framework install, hosted service, or credential.


The optional live example uses `python -m pip install llama-index llama-index-llms-huggingface-api`. Optional Phoenix tracing uses `python -m pip install arize-phoenix-otel`.

## Complete implementation

Run the next cell to build the complete offline demonstration.

In [1]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class StartEvent:
    message: str

@dataclass(frozen=True)
class ClassifyEvent:
    message: str

@dataclass(frozen=True)
class RespondEvent:
    category: str
    message: str

@dataclass(frozen=True)
class RetryEvent:
    message: str

@dataclass(frozen=True)
class StopEvent:
    result: str

@dataclass
class WorkflowState:
    trace: list = field(default_factory=list)
    attempts: int = 0
    category: str = ""
    response: str = ""

def start_step(event, state):
    state.trace.append("start")
    return ClassifyEvent(event.message)

def classify_step(event, state):
    state.trace.append("classify")
    state.attempts += 1
    normalized_message = event.message.casefold()
    if "gala" in normalized_message or state.attempts >= 2:
        state.category = "schedule"
        return RespondEvent(state.category, event.message)
    return RetryEvent(event.message + " gala")

def retry_step(event, state):
    return ClassifyEvent(event.message)

def respond_step(event, state):
    state.trace.append("respond")
    if event.category == "schedule":
        state.response = "The gala starts at 19:00."
    else:
        state.response = "No matching response."
    return StopEvent(state.response)

def run_workflow(message):
    state = WorkflowState()
    event = StartEvent(message)
    while not isinstance(event, StopEvent):
        if isinstance(event, StartEvent):
            event = start_step(event, state)
        elif isinstance(event, ClassifyEvent):
            event = classify_step(event, state)
        elif isinstance(event, RetryEvent):
            event = retry_step(event, state)
        elif isinstance(event, RespondEvent):
            event = respond_step(event, state)
        else:
            raise TypeError("unsupported event")
        if state.attempts > 2:
            raise RuntimeError("workflow exceeded its bounded classification loop")
    state.trace.append("stop")
    return event, state

workflow_result, workflow_state = run_workflow("When is the gala?")
workflow_trace = workflow_state.trace

retry_result, retry_state = run_workflow("When does it begin?")

## Deterministic assertions

Run these checks to verify the lesson without credentials, network access, or third-party packages.

In [2]:
assert isinstance(workflow_result, StopEvent)
assert workflow_result.result == "The gala starts at 19:00."
assert workflow_state.category == "schedule"
assert workflow_state.attempts == 1
assert workflow_trace == ["start", "classify", "respond", "stop"]
assert retry_state.attempts == 2
assert retry_state.trace == ["start", "classify", "classify", "respond", "stop"]
assert retry_result.result == "The gala starts at 19:00."

print("Offline assertions passed.")

Offline assertions passed.


## Results interpretation

Typed events make valid transitions visible, steps isolate state changes, and the retry branch returns a new classification event. The attempt limit guarantees termination even when classification initially lacks enough context.

## Credentials

The offline state machine uses no credentials. The optional LlamaIndex workflow checks `HF_TOKEN` before framework imports and never displays it. Phoenix imports occur only when `PHOENIX_API_KEY` is configured.

In [3]:
import os

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    print("SKIPPED: Set HF_TOKEN to run this optional LlamaIndex example.")
else:
    try:
        from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
        from llama_index.core.workflow import Context, StartEvent, StopEvent, Workflow, step
    except ImportError:
        print("SKIPPED: llama-index is not installed; run `python -m pip install llama-index llama-index-llms-huggingface-api`.")
    else:
        phoenix_api_key = os.getenv("PHOENIX_API_KEY")
        if phoenix_api_key:
            try:
                from phoenix.otel import register
            except ImportError:
                print("SKIPPED: Phoenix tracing is not installed; run `python -m pip install arize-phoenix-otel`.")
            else:
                tracer_provider = register(
                    project_name="hf-agents-course-llama-index",
                    endpoint="https://app.phoenix.arize.com/v1/traces",
                    headers={"api_key": phoenix_api_key},
                    auto_instrument=True,
                )
                print("Phoenix tracing configured.")
        else:
            print("SKIPPED: PHOENIX_API_KEY is absent; tracing remains disabled.")

        llm = HuggingFaceInferenceAPI(
            model_name="Qwen/Qwen2.5-7B-Instruct",
            token=hf_token,
            task="text-generation",
        )
        class LiveScheduleWorkflow(Workflow):
            @step
            async def answer(self, context: Context, event: StartEvent) -> StopEvent:
                await context.store.set("stage", "respond")
                completion = await llm.acomplete(
                    "Answer exactly: The gala starts at 19:00."
                )
                return StopEvent(result=str(completion))

        live_workflow = LiveScheduleWorkflow(timeout=30, verbose=False)
        print("Configured a live typed workflow; call `await live_workflow.run()` to use the hosted model.")

SKIPPED: PHOENIX_API_KEY is absent; tracing remains disabled.
Configured a live typed workflow; call `await live_workflow.run()` to use the hosted model.


## Completion checklist

- [ ] Followed each typed event transition.
- [ ] Exercised the retry branch and bounded loop.
- [ ] Verified state and both workflow traces.

## Attribution

This typed state-machine example is original and informed by the official [LlamaIndex Workflows documentation](https://developers.llamaindex.ai/python/workflows/). It does not reproduce an upstream notebook. Notebook code carries the Apache-2.0 metadata license.